In [ ]:
#| label: fig1cell
from pathlib import Path
import numpy as np, pandas as pd
import plotly.graph_objects as go

PXIN, PT = 96, 96 / 72
px = lambda i: round(i * PXIN); pt = lambda p: p * PT

T1_SCALE, N_SLICES, MIN_N = 1000.0, 46, 3
C_OD, C_OS = 'rgb(34,139,94)', 'rgb(59,130,246)'
AX = dict(color='black', linecolor='black', showline=True, mirror=False, showgrid=False,
          zeroline=False, ticks='outside', tickcolor='black', fixedrange=True)

prof = pd.read_csv('data/data_profile.csv')
slice_cols = [f'T1_slice_{i:02d}' for i in range(N_SLICES)]
mm = np.arange(N_SLICES) + 0.5

fig = go.Figure()
for eye, col in [('OD', C_OD), ('OS', C_OS)]:
    M = prof.loc[prof.Eye == eye, slice_cols].to_numpy() * T1_SCALE
    if M.size == 0:
        continue
    faint = col.replace('rgb', 'rgba').replace(')', ',0.2)')
    for row in M:
        fig.add_scatter(x=mm, y=row, mode='lines', line=dict(color=faint, width=0.8),
                        hoverinfo='skip', showlegend=False, connectgaps=False)
    xs, mean, sd = [], [], []
    for j in range(N_SLICES):
        v = M[:, j]; v = v[~np.isnan(v)]
        if len(v) < MIN_N:
            continue
        xs.append(mm[j]); mean.append(v.mean()); sd.append(v.std(ddof=1))
    od = eye == 'OD'
    fig.add_scatter(x=xs, y=mean, mode='lines+markers', name='OD (Right)' if od else 'OS (Left)',
        line=dict(color=col, width=2),
        marker=dict(size=10, color='white' if od else col,
                    line=dict(color=col if od else 'white', width=1.5)),
        error_y=dict(type='data', array=sd, visible=True, color=col, thickness=1.2, width=4))

fig.update_layout(autosize=True, paper_bgcolor='white', plot_bgcolor='white',
    dragmode=False, font=dict(color='black', family='DejaVu Sans, Arial, sans-serif', size=pt(12)),
    title=dict(text='T₁ as a function of position along the ON', x=0.5, xanchor='center', font=dict(size=pt(14))),
    margin=dict(l=px(0.9), r=px(0.1), t=px(0.6), b=px(0.6)),
    xaxis=dict(**AX, title=dict(text='Position along ON (mm)', standoff=5), range=[0, 15]),
    yaxis=dict(**AX, title=dict(text='T₁ (ms)'), range=[500, 1800]),
    legend=dict(x=0.02, y=0.98, xanchor='left', yanchor='top',
                bgcolor='rgba(255,255,255,0)', borderwidth=0, font=dict(size=pt(12))))
fig.show(config={'responsive': True})